# Benchmark RAG Chunking (BM25 Baseline)
Notebook này nhằm mục đích thử nghiệm và so sánh độ hiệu quả (Recall@K) của thuật toán BM25 trên 2 kỹ thuật Chunking:
1. **Sentence-based Chunking (Không có Context)**
2. **Contextual Chunking (Gắn ngữ cảnh First Paragraph)**
3. **True Semantic Chunking (Sử dụng Qwen3-Embedding-8B)**
4. **Structure Chunking (Nhận diện Markdown Headings)**

Tập dữ liệu: `data/raw/corpus_text/corpus_part_00000.parquet` (10,000 bài báo) và 50 câu hỏi từ Dev Set.


In [ ]:
# Cài đặt các thư viện cần thiết (Chạy cell này đầu tiên)
!pip install -q pyvi rank_bm25 sentence-transformers pandas pyarrow scikit-learn


In [ ]:
import pandas as pd
import json
import sys
import os
from pyvi import ViTokenizer
from rank_bm25 import BM25Okapi
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CẤU HÌNH KAGGLE & ĐƯỜNG DẪN
# ==========================================
import os
import sys

# Đổi thành True nếu bạn đang chạy trên Kaggle
IS_KAGGLE = False

if IS_KAGGLE:
    # Lấy token từ Kaggle Secrets để clone repo Private (nếu cần)
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    
    # Clone repo về môi trường Kaggle
    !git clone https://{github_token}@github.com/ptquanh/R2AI-Stage-3-2026.git /kaggle/working/R2AI-Stage-3-2026
    
    BASE_DIR = "/kaggle/working/R2AI-Stage-3-2026"
    SRC_DIR = f"{BASE_DIR}/src/preprocessing"
    
    # Thư mục Dataset parquet bạn upload lên Kaggle
    PARQUET_DIR = "/kaggle/input/ten-dataset-parquet-cua-ban"
else:
    BASE_DIR = ".."
    SRC_DIR = "../src/preprocessing"
    PARQUET_DIR = f"{BASE_DIR}/data/raw/corpus_text"

# Import Chunker
sys.path.append(os.path.abspath(SRC_DIR))
from chunker import Chunker


In [ ]:
# Tải Dev Queries và Qrels
with open(f'{BASE_DIR}/data/dev_set/dev_queries.json', 'r', encoding='utf-8') as f:
    dev_queries = json.load(f)
    
qrels = pd.read_csv(f'{BASE_DIR}/data/dev_set/dev_qrels_synthetic_data.csv')

# Tải Corpus: Thay vì quét toàn bộ parquet (có thể chưa cào xong), 
# ta lấy trực tiếp văn bản đáp án từ dev_ground_truth.json
with open(f'{BASE_DIR}/data/dev_set/dev_ground_truth.json', 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

true_docs = []
for item in ground_truth:
    for chunk in item['relevant_chunks']:
        true_docs.append({
            'id': str(chunk['doc_id']),
            'text': chunk['chunk_text'],
            'url': ''
        })

df_true = pd.DataFrame(true_docs).drop_duplicates(subset=['id'])

# Lấy 10,000 bài báo từ file parquet đầu tiên để làm dữ liệu nhiễu (noise)
# (Trên Kaggle, file này nằm trong PARQUET_DIR)
df_temp = pd.read_parquet(f'{PARQUET_DIR}/corpus_part_00000.parquet')
df_temp['id'] = df_temp['id'].astype(str)
df_noise = df_temp[~df_temp['id'].isin(df_true['id'])].head(10000)

df_corpus = pd.concat([df_true, df_noise]).drop_duplicates(subset=['id']).reset_index(drop=True)

print(f"Đã tạo Benchmark Corpus với tổng {len(df_corpus)} bài báo (Gồm {len(df_true)} đáp án và {len(df_noise)} nhiễu)")


## Hàm Chạy BM25 và Đánh giá (Recall@K)

In [ ]:
def tokenize(text):
    # Tách từ tiếng Việt bằng PyVi
    return ViTokenizer.tokenize(str(text).lower()).split()

def evaluate_bm25(chunks_df):
    print("Đang Tokenize chunks cho BM25 (chờ chút nhé)...")
    tokenized_corpus = [tokenize(text) for text in chunks_df['content']]
    bm25 = BM25Okapi(tokenized_corpus)
    
    recall_at_5 = 0
    recall_at_10 = 0
    total = len(dev_queries)
    
    for q in dev_queries:
        query_id = q['id']
        query_text = q['query']
        # Tìm doc_id thực sự là đáp án
        true_doc_id = str(next(item['relevant_docs'][0] for item in ground_truth if item['id'] == query_id))
        
        # Tìm kiếm BM25
        tokenized_query = tokenize(query_text)
        doc_scores = bm25.get_scores(tokenized_query)
        
        # Lấy Top 10 chunks
        top_10_indices = np.argsort(doc_scores)[::-1][:10]
        top_10_doc_ids = [chunks_df.iloc[idx]['doc_id'] for idx in top_10_indices]
        
        if true_doc_id in top_10_doc_ids[:5]:
            recall_at_5 += 1
        if true_doc_id in top_10_doc_ids:
            recall_at_10 += 1
            
    print(f"=============================")
    print(f"Recall@5:  {recall_at_5/total:.2%} ({recall_at_5}/{total})")
    print(f"Recall@10: {recall_at_10/total:.2%} ({recall_at_10}/{total})")
    print(f"=============================")


## Kịch bản 1: Không có Context (Tắt Contextual Injection)

In [ ]:
chunks_no_context = []
chunker = Chunker(strategy="sentence", max_tokens=256, token_overlap=50)

print("Đang chunking (Không Context)...")
for _, row in df_corpus.iterrows():
    text = str(row['text']) if row['text'] else ""
    if text:
        res = chunker.chunk_document(str(row['id']), text)
        # Giả lập tắt Context: Bỏ đi prefix Context nếu có
        for c in res:
            if '[Context:' in c['content']:
                c['content'] = c['content'].split(']', 1)[-1].strip()
        chunks_no_context.extend(res)

df_chunks_no_ctx = pd.DataFrame(chunks_no_context)
print(f"Tạo được {len(df_chunks_no_ctx)} chunks.")
evaluate_bm25(df_chunks_no_ctx)


## Kịch bản 2: Có Contextual Injection (Kỹ thuật của Anthropic)

In [ ]:
chunks_with_context = []
chunker = Chunker(strategy="contextual", max_tokens=256, token_overlap=50)

print("Đang chunking (CÓ Context)...")
for _, row in df_corpus.iterrows():
    text = str(row['text']) if row['text'] else ""
    if text:
        res = chunker.chunk_document(str(row['id']), text)
        chunks_with_context.extend(res)

df_chunks_with_ctx = pd.DataFrame(chunks_with_context)
print(f"Tạo được {len(df_chunks_with_ctx)} chunks.")
evaluate_bm25(df_chunks_with_ctx)


## Kịch bản 3: True Semantic Chunking (Dùng mô hình Sentence-Transformers)

In [ ]:
chunks_semantic = []
# Khởi tạo Semantic Chunker với model Qwen3-Embedding-8B (thoả mãn <15B, public trước 01/08/2026)
try:
    chunker_sem = Chunker(strategy="semantic", max_tokens=256, token_overlap=50)
    
    print("Đang chunking (Semantic Embedding - quá trình này sẽ mất thời gian hơn vì phải tải model)...")
    for _, row in df_corpus.iterrows():
        text = str(row['text']) if row['text'] else ""
        if text:
            chunks = chunker_sem.chunk_document(doc_id=str(row['id']), text=text, url=row['url'])
            chunks_semantic.extend(chunks)
            
    df_chunks_sem = pd.DataFrame(chunks_semantic)
    print(f"Tạo được {len(df_chunks_sem)} semantic chunks.")
    
    evaluate_bm25(df_chunks_sem)
except Exception as e:
    print(f"Lỗi khởi tạo hoặc chạy Semantic Chunker: {e}")


## Kịch bản 4: Structure Chunking (Tiêm Cấu trúc Markdown)

In [ ]:
chunks_structure = []
chunker_struct = Chunker(strategy="structure", max_tokens=256, token_overlap=50)

print("Đang chunking (Structure-aware - Tách theo Markdown Headings)...")
for _, row in df_corpus.iterrows():
    text = str(row['text']) if row['text'] else ""
    if text:
        chunks = chunker_struct.chunk_document(doc_id=str(row['id']), text=text, url=row['url'])
        chunks_structure.extend(chunks)
        
df_chunks_struct = pd.DataFrame(chunks_structure)
print(f"Tạo được {len(df_chunks_struct)} structure chunks.")

evaluate_bm25(df_chunks_struct)
